# Evaluate a trained VLG-CBM

The counterpart to `evaluate_my_cbm.ipynb` in the Label-free-CBM repo. Sections 1-11 are
the same analyses in the same order, so the two notebooks can be read side by side; all
logic lives in `vlgcbm_analysis.py` and this notebook is a driver.

Sections 12-13 have no LF-CBM equivalent. They use the Grounding DINO annotations to ask
whether a concept is actually *in* the image, which is the question LF-CBM cannot pose.

Run from the repo root. Requires `DATASET_FOLDER` and `VLGCBM_BIOCLIP_CKPT`, set below.

In [ ]:
import os
os.environ.setdefault("DATASET_FOLDER", "/workspace/VLG-CBM/datasets")
os.environ.setdefault("VLGCBM_BIOCLIP_CKPT", "/workspace/models/bioclip/open_clip_pytorch_model.bin")
os.environ["TORCH_DEVICE_BACKEND_AUTOLOAD"] = "0"

import torch
import vlgcbm_analysis as va

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("kernel :", os.environ.get("CONDA_PREFIX") or "unknown -- must be Python (vlgcbm)")
print("\nmodels with a finished run:", ", ".join(va.list_models("birds525")))

## Config

Two variables. `MODEL` is any name printed above; `DATASET` selects the dataset.

`SPLIT="val"` is VLG-CBM's held-out **test** set -- the one `train_cbm.py` reports test
accuracy on. The validation split used during training is carved out of `<dataset>_train`
and has no name here.

In [ ]:
MODEL   = "bioclip"      # <-- swap me
DATASET = "birds525"     # <-- and me
SPLIT   = "val"

run = va.load_run(va.run_dir(MODEL, DATASET))
run

## 1. Accuracy

`evaluate` keeps per-example predictions and concept activations in dataset order, so
every plot below refers to the same indices.

In [ ]:
# Cached to <run>/eval_val.pt on first call, so re-running this cell is instant.
# Pass cache=False to force a recompute.
res = va.evaluate(run, split=SPLIT)
print(res)
print("accuracy: {:.2f}%".format(res.accuracy * 100))
print("wrong:    {} of {}".format(len(res.wrong_indices), len(res.labels)))

## 2. Compare models

Accuracy alongside sparsity. Sparsity is the real caveat in this table: VLG-CBM's own
argument is that accuracy is only comparable at a fixed Number of Effective Concepts, so
treat `concepts_per_class` as the axis that makes the accuracy column meaningful. For the
paper's ANEC metric proper, use `sparse_evaluation.py`.

In [ ]:
va.compare(va.find_runs(), split=SPLIT)

## 3. Wrong predictions

Each row: the misclassified image, and the concepts that pushed the model toward the
wrong class.

In [ ]:
va.plot_wrong_predictions(run, res, n=5, top_k=8, seed=0)

## 4. Single example

The per-decision bar plot, for any index in the eval set.

In [ ]:
va.explain_example(run, idx=20, split=SPLIT)

## 5. Sankey: concept -> class

Final-layer weights as concept -> class flows, styled after Figure 3 of the Label-free CBM
paper. Section 7 tells you which two classes this model actually confuses -- those are the
pair worth contrasting here.

In [ ]:
print("first 10 classes:", run.classes[:10])
va.sankey_static(run, [run.classes[0], run.classes[1]],
                 weight_cutoff=0.05, max_per_class=12,
                 save_path="figures/sankey.png")

In [ ]:
# The paper's Figure 3 layout -- one panel per class pair, side by side.
# Fill in the pairs from section 7 once you know what this model confuses.
#
# va.sankey_panels(run, [("abbotts babbler", "abbotts booby")],
#                  weight_cutoff=0.05, max_per_class=12,
#                  save_path="figures/sankey_panels.png")

## 6. Concept activation heatmap

Dataset-level view of which concepts fire, complementing the per-example bars.

In [ ]:
va.concept_heatmap(run, res, n_examples=30, n_concepts=20)

## 7. Most confused class pairs

Where to point the Sankey next.

In [ ]:
for pair in va.confused_pairs(run, res, k=10):
    print("{:4d}x  {}  ->  {}".format(pair["count"], pair["true"], pair["pred"]))

## 8. Result collages

Three sheets of 10 examples each, saved next to the run: `random`, `confident_wrong`,
`least_confident`. The confident-and-wrong sheet is where the interesting failures are.

In [ ]:
paths = va.save_collages(run, res, n=10, seed=0)
paths

In [ ]:
va.result_collage(run, res, mode="confident_wrong", n=10)

## 9. Per-class accuracy

Left: how per-class accuracy is distributed. Right: the weakest classes by accuracy.

In [ ]:
rows = va.plot_class_accuracy(run, res, worst_k=25,
                              save_path="figures/class_accuracy.png")
print("\nworst 10:")
for r in rows[:10]:
    print("  {:5.1f}%  {:3d}/{:<3d}  {}".format(
        r["accuracy"] * 100, r["correct"], r["support"], r["class"]))

## 10. Confusion matrix

The full matrix is 525x525 and almost entirely zeros, so this restricts rows to the
weakest classes.

In [ ]:
mat = va.plot_confusion(run, res, worst_k=25, save_path="figures/confusion.png")

## 11. Cross-model qualitative comparison

One figure, several rows, each a different story about where the backbones diverge.
`evaluate_many` releases each model before loading the next, so all five do not have to
be GPU-resident at once.

In [ ]:
comparison_runs = va.find_runs()
outs = va.evaluate_many(comparison_runs, split=SPLIT, keep_concept_acts=True)
for o in outs:
    print(o)

In [ ]:
va.story_figure(outs, split=SPLIT, top_concepts=2,
                flag_mode="sufficiency",      # or "concentration", or None to disable
                save_path="figures/qualitative_comparison.png")

### The amber `OK*` cells

Green = correct and the displayed concepts hold up. Amber = correct prediction, but the
displayed concepts do not by themselves account for it -- a proxy, not a verdict. Inspect
before trusting.

In [ ]:
for o in outs:
    ok = o.correct.nonzero(as_tuple=True)[0]
    conc = [va.explanation_concentration(o, int(j), k=2) for j in ok[:200]]
    flagged = sum(va.flag_explanation(o, int(j), k=2, mode="sufficiency") for j in ok[:200])
    import numpy as np
    print("%-34s median top-2 share %.3f | sufficiency flags %d/%d" % (
        o.name, float(np.median(conc)) if conc else float("nan"), flagged, len(ok[:200])))

## 12. Grounded explanation  *(VLG-CBM only)*

The bar plot next to the image with Grounding DINO's boxes drawn on.

This is the question LF-CBM cannot ask. The bars say which concepts drove the prediction;
the boxes say whether those concepts were ever found in the image. A concept carrying a
large positive contribution **with no box** is acting as a class prior rather than as
evidence -- which is exactly what a non-localisable concept ("birdsong", "oscine") looks
like from the inside. Such concepts are marked `(no box)`.

In [ ]:
va.explain_with_boxes(run, idx=20, split=SPLIT, top_k=8)

In [ ]:
# The same view on a mistake -- usually more informative than a success.
va.explain_with_boxes(run, idx=int(res.wrong_indices[0]), split=SPLIT, top_k=8)

## 13. Concept grounding  *(VLG-CBM only)*

Per-concept AUC between the CBL's activation and the annotation label it was trained to
reproduce, on held-out data.

Read low AUC two ways: the CBL failed to learn that concept, **or** the concept is not
recoverable from the image at all. For birds525 the bottom of this list is where the
abstract concepts collect -- the ones Grounding DINO grounded to something arbitrary and
that survived filtering only because the rule drops a concept solely when it has *zero*
detections across the entire train set (101 of 2,012 here).

In [ ]:
rows = va.concept_agreement(run, res, split=SPLIT)
print("scored {} concepts".format(len(rows)))
print("\nleast grounded:")
for r in rows[:15]:
    print("  AUC {:.3f}  n={:5d}  {}".format(r["auc"], r["support"], r["concept"]))
print("\nbest grounded:")
for r in rows[-10:][::-1]:
    print("  AUC {:.3f}  n={:5d}  {}".format(r["auc"], r["support"], r["concept"]))

In [ ]:
va.plot_concept_agreement(run, rows=rows, k=20,
                          save_path="figures/concept_grounding.png")